# 04 — Where a gain could have shown, and why none did

Two questions that a single budget cannot answer. Could the experiment have seen a gain at
all, given how little headroom the task leaves? And if the pseudo-labels carry information,
what happens to it between the pre-training and the fine-tuning?

Each mechanism is read against its own control, never against the plain baseline.

In [1]:
import numpy as np
import pandas as pd

from mri_semisupervised.config import EXPERIMENTS_DIR
from mri_semisupervised.protocol.uncertainty import paired_difference

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

per_fold = pd.read_parquet(EXPERIMENTS_DIR / "corrected" / "per_fold.parquet")
predictions = pd.read_parquet(EXPERIMENTS_DIR / "corrected" / "predictions.parquet")
folds = pd.read_parquet(EXPERIMENTS_DIR / "corrected" / "folds.parquet")

#: The five metrics every table of these notebooks reports, in one order.
headline = ["roc_auc", "pr_auc", "recall_positive", "f1_macro", "accuracy"]

## 1. Where the experiment could see anything at all

A null result is only worth reading if the experiment could have shown a gain. The
supervised baseline reaches 0.966 with eight folds out of twenty-five already at 1.000, so
what is left to win is about the size of the noise. The arms were therefore rerun at
smaller labelling budgets, where a semi-supervised method has room to help.

In [2]:
budgets = {}
for name, size in [("budget-10", 10), ("budget-20", 20), ("budget-40", 40), ("corrected", 59)]:
    path = EXPERIMENTS_DIR / name / "per_fold.parquet"
    if path.exists():
        budgets[size] = pd.read_parquet(path).pivot(index="fold", columns="arm", values="roc_auc")

rows = []
for size, pivot in sorted(budgets.items()):
    row = {"labels": size}
    row.update({arm: round(pivot[arm].mean(), 3) for arm in pivot.columns})
    if {"semi_supervised", "permuted_control"} <= set(pivot.columns):
        out = paired_difference(
            pivot["semi_supervised"].to_numpy(), pivot["permuted_control"].to_numpy()
        )
        row["semi - control"] = f"{out['mean_difference']:+.3f} (p={out['p_value']:.2f})"
    rows.append(row)
pd.DataFrame(rows)

,labels,permuted_control,semi_supervised,supervised,semi - control,semi_supervised_confident
0,10,0.920,0.941,0.941,+0.022 (p=0.12),NaN
1,20,0.921,0.926,0.952,+0.005 (p=0.71),NaN
2,40,0.947,0.947,0.957,+0.000 (p=0.97),NaN
3,59,0.952,0.958,0.966,+0.006 (p=0.57),0.945


Two things this table says that a single budget could not. The semi-supervised arm never
beats the plain baseline, at any budget. And the permuted control sits *below* that baseline
everywhere — so the pre-training phase costs something on its own, and real pseudo-labels
recover part of that cost without ever turning it into a gain.

## 2. Two hypotheses about why, each with its own control

Two explanations for the null result are worth testing. Maybe the
pre-training is simply **forgotten** — 246 steps, then a fine-tuning that converges in two
to four epochs. Maybe the labels come from the **wrong source** — a k-means on ImageNet
embeddings, and not from the decision function being optimised.

`semi_supervised_joint` keeps the pseudo-label loss present at every step;
`self_training` builds its labels from its own first pass. Each is read against its own
control, never against the plain baseline.

In [3]:
stage_b = EXPERIMENTS_DIR / "corrected-stageb" / "per_fold.parquet"
if stage_b.exists():
    mech = pd.read_parquet(stage_b).pivot(index="fold", columns="arm", values="roc_auc")
    joined = pd.concat(
        [pivot_full := per_fold.pivot(index="fold", columns="arm", values="roc_auc"), mech], axis=1
    )
    rows = []
    for arm, control in [
        ("semi_supervised_joint", "joint_permuted_control"),
        ("self_training", "self_training_control"),
    ]:
        against_control = paired_difference(joined[arm].to_numpy(), joined[control].to_numpy())
        against_baseline = paired_difference(
            joined[arm].to_numpy(), joined["supervised"].to_numpy()
        )
        rows.append(
            {
                "arm": arm,
                "vs its control": f"{against_control['mean_difference']:+.3f} "
                f"(p={against_control['p_value']:.3f})",
                "vs supervised": f"{against_baseline['mean_difference']:+.3f} "
                f"(p={against_baseline['p_value']:.3f})",
            }
        )
    display(pd.DataFrame(rows))

,arm,vs its control,vs supervised
0,semi_supervised_joint,+0.033 (p=0.008),-0.010 (p=0.113)
1,self_training,+0.015 (p=0.256),-0.003 (p=0.726)


Joint training beats its control significantly and still loses to the baseline. Both facts
are needed: the control sits at 0.922 because training on shuffled labels at every step is
harmful, so the win against it measures that harm, and not a gain. A significant p-value
against the correct control can still mean the opposite of what it looks like.